In [29]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from gspread_pandas import Spread, conf

In [37]:
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1aXobnCl4QQ-hQ_IWoTPo2ZEo3HM5kNwycwMBJtEDOfc/edit?gid=822921427#gid=822921427"
# sales_sheet_url = "https://docs.google.com/spreadsheets/d/1h7txdIo1ha4vp2uE8W2E4uUXgJYSG_wgxyKGpW7rYAQ/edit?gid=1442776849#gid=1442776849"


# Load the config from your service account JSON
c = conf.get_config(file_name=config_path)

# Connect using that config object
# sales_spread = Spread(sales_sheet_url, config=c)
emarath_spread = Spread(emarath_global_sheet_url, config=c)

In [38]:
# 5. Loop through and pull categorized data
all_data_list = []
all_sheets = emarath_spread.sheets


for sheet in all_sheets:
    sheet_name = sheet.title
    
    # Skip dashboard and administrative tabs
    if sheet_name in ["MAIN DASHBOARD", "CRM", "Dropdowns", "Sheet1", "DASHBOARD", "20"]:
        continue
    
    try:
        df = emarath_spread.sheet_to_df(sheet=sheet_name, index=None, header_rows=1)
        
        # Check for necessary columns: DATE, STATUS, and LANGUAGE
        if not df.empty and all(col in df.columns for col in ['DATE', 'LANGUAGE']):
  
            
            if not df.empty:
                df['Name of BDE'] = sheet_name
                # Keep only relevant columns for the final report
                all_data_list.append(df[['DATE', 'Name of BDE', 'LANGUAGE']])
            
    except Exception as e:
        print(f"Skipping {sheet_name}: {e}")

# Combine and Process
if all_data_list:
    master_df = pd.concat(all_data_list, ignore_index=True)
    
    # Standardize Date format
    master_df['Date'] = pd.to_datetime(master_df['DATE'], dayfirst=True, errors='coerce').dt.date

    

    # --- 2. LANGUAGE COUNT PIVOT ---
    language_df = master_df.pivot_table(
        index=['Date', 'Name of BDE'], 
        columns='LANGUAGE', 
        aggfunc='size', 
        fill_value=0
    )
    final_report = language_df
    final_report = pd.concat([language_df], axis=1).fillna(0).astype(int)

    print("\n--- DAILY BDE PERFORMANCE (STATUS & LANGUAGE) ---")
    # print(final_report)
    
    # Export to Excel
    final_report.to_excel("./Output_Data/BDE_Full_Performance_Report.xlsx")
else:
    print("No data found matching the criteria.")


--- DAILY BDE PERFORMANCE (STATUS & LANGUAGE) ---


In [39]:
final_report


LANGUAGE                    ARABIC  BANGLA  ENGLISH  HINDI  MALAYALAM  TAMIL  \
Date       Name of BDE                                                         
2026-02-04 ADHIL         5       0       0        1      2          7      2   
           AMEEN        89       0       0        0      0          0      0   
           AMINA        21       4       0        1      1          7      2   
           ARUN         29       2       0        1      0          3      0   
           BURHANA      20       5       0        8      4          2      1   
...                     ..     ...     ...      ...    ...        ...    ...   
2026-02-16 SHAMNA        6       0       0        1      8          0      0   
           SUSHANTHIKA  26       0       0       13      1          0      0   
           ZAKIYA       18       0       0        0      1          0      1   
2026-03-14 AJIN          4       0       0        0      0          0      0   
2026-05-05 RAHIB         1       0       0        0      0          0      0   

LANGUAGE                URDU  Urdu.  
Date       Name of BDE               
2026-02-04 ADHIL           0      0  
           AMEEN           0      0  
           AMINA           0      0  
           ARUN            0      0  
           BURHANA         0      0  
...                      ...    ...  
2026-02-16 SHAMNA          0      0  
           SUSHANTHIKA     0      0  
           ZAKIYA          0      0  
2026-03-14 AJIN            0      0  
2026-05-05 RAHIB           0      0  

[157 rows x 9 columns]

In [35]:
# # Save the final combined report to an Excel file
# file_name = "./Output_Data/tag_wise_Report.xlsx"
# final_combined_report.to_excel(file_name, index=True)
# print(f"Success! The report has been saved as: {file_name}")

In [36]:

# Dest_sheet_url = "https://docs.google.com/spreadsheets/d/1PqMNtFU0bas_BGYFhIYWojO2petW-TOGxeRB2OWnF08/edit?pli=1&gid=788610855#gid=788610855"
# Dest_tab_name = "Tag_Wise_Report"

# try:
#     dest_spread = Spread(Dest_sheet_url, config=c)
#     dest_spread.open_sheet(Dest_tab_name, create=False)

#     export_df = final_combined_report.reset_index()

 
#     export_df['Date'] = export_df['Date'].astype(str)
#     export_df.loc[export_df['Date'].duplicated(), 'Date'] = ""

#     existing_df = dest_spread.sheet_to_df()
#     is_empty = existing_df.empty

#     current_data_rows = len(existing_df)

#     start_row = 1 if is_empty else current_data_rows + 2

#     # 4. Push to sheet
#     dest_spread.df_to_sheet(
#         export_df,          
#         index=False, 
#         replace=False, 
#         headers=is_empty,     
#         start=f'A{start_row}' 
#     )

#     print(f"Cleaned report appended to '{Dest_tab_name}' starting at row {start_row}.")

# except Exception as e:
#     print(f"Detailed Error: {e}")